Implement Scaled Dot-Product Attention and a Mini Transformer Encoder 
You are required to implement the core components of the Transformer architecture 
using PyTorch. The assignment focuses on understanding how attention works inside a 
Transformer. 
Problem Statement 
Implement a simple Transformer Encoder from scratch and test it on a small text 
classification task. 

You should implement the following: 
1. Scaled Dot-Product Attention  
2. Multi-Head Self-Attention  
3. Positional Encoding  
4. Feed Forward Network  
5. Transformer Encoder Block  
6. A simple classifier using the Transformer Encoder  

Dataset 
Use any small dataset, such as:

• IMDB small subset  
• AG News subset  
• SMS Spam Detection dataset  
• A custom dataset with 2 classes, for example positive/negative sentences  

You may also create a small toy dataset manually.

<div align="center">
----------------------------------------------------------------------------------------------XXXXXXXXXXXXXXXXXXXXXX----------------------------------------------------------------------------------------------
</div>

In [19]:
# import kagglehub
# path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
# print("Path to dataset files:", path)

In [29]:
#import necessary libraries
import pandas as pd
from collections import Counter

In [30]:
DATASET_PATH = r'D:\machine_learning\datasets\sms_spam_detection'
df = pd.read_csv(DATASET_PATH + '\spam.csv', encoding='latin-1') #utf-8 is failing for some reason. okay there is something wrong in row number 2, escape character probably
display(df)

df = df[['v1', 'v2']] 
df.columns = ['label', 'text']

# replacing labels with 1 and 0, with spam being 1 and ham being 0
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# final dataset for training/testing
display(df)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Sumed\AppData\Local\Temp\ipykernel_10452\2225014558.py:2: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(DATASET_PATH + '\spam.csv', encoding='latin-1') #utf-8 is failing for some reason. okay there is something wrong in row number 2, escape character probably


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


1. Preprocess the text data:  
o Tokenize sentences  
o Convert words into integer IDs  
o Pad sequences to the same length

In [31]:
#tokenizing process
def tokenize(text):
    return text.lower().split()
df['tokens'] = df['text'].apply(tokenize)
display(df[['text', 'tokens']].head())


word_counter = Counter()

for tokens in df['tokens']:
    word_counter.update(tokens)

# create vocabulary
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}

# integer IDs
for word in word_counter:
    vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

,text,tokens
0,"Go until jurong point, crazy.. Available only ...","[go, until, jurong, point,, crazy.., available..."
1,Ok lar... Joking wif u oni...,"[ok, lar..., joking, wif, u, oni...]"
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,U dun say so early hor... U c already then say...,"[u, dun, say, so, early, hor..., u, c, already..."
4,"Nah I don't think he goes to usf, he lives aro...","[nah, i, don't, think, he, goes, to, usf,, he,..."


Vocabulary Size: 13498
